In [1]:
!pip install pyarrow==2 awswrangler

     |████████████████████████████████| 17.7 MB 17.0 MB/s            
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 6.0.1
    Uninstalling pyarrow-6.0.1:
      Successfully uninstalled pyarrow-6.0.1


In [2]:
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import random
import awswrangler as wr

# six month

In [3]:
sql = f"""
SELECT 
        dp.product_line as sub_product_line,   
        dp.henry_category_1,
        -- store
        fsoo.id_store,
        fsoo.store_name,
        fsoo.id_shop,
        CASE WHEN cast(fsoo.id_shop as bigint) = 1 THEN 'TH'
                                    WHEN cast(fsoo.id_shop as bigint) = 2 THEN 'SG'
                                    WHEN cast(fsoo.id_shop as bigint) = 5 THEN 'ID'
                                    WHEN cast(fsoo.id_shop as bigint) = 11 THEN 'MY'
                                    ELSE null
                                    END as id_shop_name
        -- gross revenue
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 THEN gross_revenue_usd ELSE 0 END) as gross_revenue_usd_week7

        -- revenue
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 THEN revenue_usd ELSE 0 END) as revenue_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 THEN revenue_usd ELSE 0 END) as revenue_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 THEN revenue_usd ELSE 0 END) as revenue_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 THEN revenue_usd ELSE 0 END) as revenue_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 THEN revenue_usd ELSE 0 END) as revenue_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 THEN revenue_usd ELSE 0 END) as revenue_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 THEN revenue_usd ELSE 0 END) as revenue_usd_week7
        
        -- item discount
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 AND 
(fsoo.promotion_name like '%Item Discount%' or fsoo.promotion_name like '%itemdiscount%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as item_discount_usd_week7
            
        -- voucher discount
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=0 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=6 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week1
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=7 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=13 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week2
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=14 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=20 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week3
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=21 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=27 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week4
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=28 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=34 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week5
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=35 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=41 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week6
        ,SUM(CASE WHEN DATE_DIFF('day',dp.date_released,fsoo.transaction_date)>=42 AND DATE_DIFF('day',dp.date_released,fsoo.transaction_date)<=48 AND 
(fsoo.promotion_name like '%Voucher%' or fsoo.promotion_name like '%voucher%') THEN fsoo.gross_revenue_usd - fsoo.revenue_usd ELSE 0 END) as voucher_discount_usd_week7
        
        
           
   
        FROM dwh.dim_product dp


        -----------------------------------JOIN fsoo ---------------------------------------------
        --  net unit sold, gross unit sold, gross rev, rev by id_product_attribute by store  --
        LEFT JOIN
            (SELECT id_product_attribute,
                id_product,
                fso.id_store,
                date(fso.transaction_time) as transaction_date,
                fso.id_shop,
                fso.promotion_name as promotion_name,
                fso.transaction_type,
                CASE WHEN ds.store_name = 'CentralWorld' THEN 'Central World'
                     WHEN ds.store_name = 'Interchange21' THEN 'Interchange 21' 
                     WHEN ds.store_name = 'All Seasons' THEN 'All Seasons Place' 
                     ELSE ds.store_name END as store_name,
                COALESCE(SUM(CASE WHEN transaction_type ='Return' THEN -product_quantity ELSE product_quantity END),0) as net_units_sold,
                COALESCE(SUM(CASE WHEN transaction_type ='Sale' THEN product_quantity ELSE 0 END),0) as gross_units_sold,
                COALESCE(SUM(CASE WHEN transaction_type ='Sale' THEN gross_revenue_usd ELSE 0 END),0) as gross_revenue_usd,
                COALESCE(SUM(CASE WHEN transaction_type ='Sale' THEN revenue_usd ELSE 0 END),0) as revenue_usd
            FROM dwh.fact_sales_offline fso
            LEFT JOIN dwh.dim_store ds on fso.id_store = ds.id_store
            WHERE store_name NOT IN ('Pomelo Men pop up','Bangna Warehouse','Silom Complex','Werk Ari','Siam Center (closed)')
            GROUP BY 1,2,3,4,5,6,7,8) fsoo 
        ON fsoo.id_product_attribute = dp.id_product_attribute 


        WHERE date_released BETWEEN DATE_TRUNC('month', NOW() - INTERVAL '6' month) AND current_date
        AND fsoo.id_store NOT IN ('39-1')
        AND dp.parent_product_line != 'Free Gift'
        AND dp.henry_category_1 NOT IN ('Accessories','Bags','Bath&Body','Beverage Container','Cosmetics','Hair','Miscellaneous','Shoes','Skin Care','Stationery')
        AND dp.brand IN ('Alita','Basics','Pomelo', 'BEET by Pomelo', 'Holiday Collection', 'Pomelo x Tex Saverio','Blackdog BKK')
        AND dp.product_cost_usd_first_order_date > 0 
        AND dp.size !=  '33'
        AND dp.release_collection_name IS NOT NULL
        AND dp.fabric_custom_name IS NOT NULL
        AND dp.original_price_th_to_usd IS NOT NULL
        GROUP BY 1,2,3,4,5,6

"""

In [4]:
df = wr.athena.read_sql_query(sql, database="dwh")

In [5]:
df[
    (df["sub_product_line"] == "Basic")
    & (df["henry_category_1"] == "Bottoms")
    & (df["id_store"] == "251")
]

,sub_product_line,henry_category_1,id_store,store_name,id_shop,id_shop_name,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,...,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7
372,Basic,Bottoms,251,EmQuartier,1,TH,270.566368,145.901063,229.71568,105.133845,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df[
    (df["sub_product_line"] == "Collection")
    & (df["henry_category_1"] == "Tops")
    & (df["id_store"] == "401")
]

,sub_product_line,henry_category_1,id_store,store_name,id_shop,id_shop_name,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,...,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7
359,Collection,Tops,401,Seacon Bangkae,1,TH,602.700391,777.044794,535.05754,389.648552,...,0.0,0.0,0.0,5.540565,0.0,1.792867,0.0,0.0,0.0,0.0


In [7]:
df

,sub_product_line,henry_category_1,id_store,store_name,id_shop,id_shop_name,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,...,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7
0,Collaboration,Outerwears,371,Siam Center,1,TH,8635.581355,2661.288500,2022.911234,842.911924,...,0.0,0.0,0.0,7.994651,0.000000,125.541041,0.000000,33.434593,0.000000,0.000000
1,Weekend,Outerwears,32,SG Nex,2,SG,1008.346722,1273.450392,1532.964600,1687.625208,...,0.0,0.0,0.0,66.479372,23.456632,10.763284,214.779065,29.680952,46.911013,2.781181
2,Collection,Tops,361,Terminal 21,1,TH,747.368626,855.792586,732.636977,744.147320,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Collaboration,Tops,361,Terminal 21,1,TH,5134.158707,2175.060165,2144.083642,1333.118251,...,0.0,0.0,0.0,77.384504,32.105109,193.804356,97.858829,30.221428,0.000000,0.000000
4,Workwear,Outerwears,301,Central Phuket,1,TH,626.274981,571.261712,372.182190,190.883889,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
703,Workwear,Bottoms,301,Central Phuket,1,TH,889.226105,515.870724,797.355830,623.586197,...,0.0,0.0,0.0,5.711502,6.219358,10.568734,7.474015,0.000000,54.845436,0.000000
704,Workwear,Outerwears,261,Central World,1,TH,2688.708719,1947.943980,1953.111102,430.643034,...,0.0,0.0,0.0,2.353456,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
705,Workwear,Outerwears,421,Central Festival Chiang Mai,1,TH,898.736853,179.894825,181.424483,239.699346,...,0.0,0.0,0.0,2.745248,0.000000,0.000000,2.394722,0.000000,2.404427,0.000000
706,Weekend,Jumpsuits & Rompers,361,Terminal 21,1,TH,0.000000,30.217564,0.000000,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [8]:
def calculate_day_no(row):
    if row["week_id"] == "week1":
        return 0
    if row["week_id"] == "week2":
        return 7
    if row["week_id"] == "week3":
        return 14
    if row["week_id"] == "week4":
        return 21
    if row["week_id"] == "week5":
        return 28
    if row["week_id"] == "week6":
        return 35
    if row["week_id"] == "week7":
        return 42

In [9]:
%%time
# unpivot 'gross_revenue_usd','revenue_usd','item_discount_usd','voucher_discount_usd'
unpivot_df = (
    pd.wide_to_long(
        df.reset_index(),
        stubnames=[
            "gross_revenue_usd",
            "revenue_usd",
            "item_discount_usd",
            "voucher_discount_usd",
        ],
        i="index",
        j="week_id",
        sep="_",
        suffix="\w+",
    )
    .reset_index(level=1)
    .reset_index(drop=True)
)

CPU times: user 54.7 ms, sys: 717 µs, total: 55.4 ms
Wall time: 70.1 ms


In [10]:
unpivot_df

,week_id,sub_product_line,id_shop,store_name,id_store,id_shop_name,henry_category_1,gross_revenue_usd,revenue_usd,item_discount_usd,voucher_discount_usd
0,week1,Collaboration,1,Siam Center,371,TH,Outerwears,8635.581355,7710.907596,0.0,7.994651
1,week1,Weekend,2,SG Nex,32,SG,Outerwears,1008.346722,831.787586,0.0,66.479372
2,week1,Collection,1,Terminal 21,361,TH,Tops,747.368626,600.022004,0.0,0.000000
3,week1,Collaboration,1,Terminal 21,361,TH,Tops,5134.158707,4297.550972,0.0,77.384504
4,week1,Workwear,1,Central Phuket,301,TH,Outerwears,626.274981,523.694880,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
4951,week7,Workwear,1,Central Phuket,301,TH,Bottoms,706.203355,559.087264,0.0,0.000000
4952,week7,Workwear,1,Central World,261,TH,Outerwears,0.000000,0.000000,0.0,0.000000
4953,week7,Workwear,1,Central Festival Chiang Mai,421,TH,Outerwears,0.000000,0.000000,0.0,0.000000
4954,week7,Weekend,1,Terminal 21,361,TH,Jumpsuits & Rompers,0.000000,0.000000,0.0,0.000000


In [17]:
discount = (
    unpivot_df.groupby(
        ["sub_product_line", "id_shop_name", "henry_category_1", "week_id"]
    )["gross_revenue_usd", "revenue_usd", "item_discount_usd", "voucher_discount_usd"]
    .median()
    .reset_index()
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:1: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  if __name__ == '__main__':


In [18]:
discount

,sub_product_line,id_shop_name,henry_category_1,week_id,gross_revenue_usd,revenue_usd,item_discount_usd,voucher_discount_usd
0,Basic,ID,Bottoms,week1,135.165571,129.316391,0.0,0.000000
1,Basic,ID,Bottoms,week2,218.077297,165.349441,0.0,0.000000
2,Basic,ID,Bottoms,week3,417.768651,293.162908,0.0,0.000000
3,Basic,ID,Bottoms,week4,333.996817,245.689323,0.0,0.000000
4,Basic,ID,Bottoms,week5,266.644736,206.069282,0.0,0.000000
...,...,...,...,...,...,...,...,...
730,Workwear,TH,Tops,week3,1311.181717,987.243534,0.0,4.411523
731,Workwear,TH,Tops,week4,884.809040,704.623662,0.0,4.349746
732,Workwear,TH,Tops,week5,652.132391,510.149222,0.0,4.167375
733,Workwear,TH,Tops,week6,257.958543,225.415301,0.0,0.000000


In [19]:
# discount
discount["discount_utilization"] = np.where(
    discount["gross_revenue_usd"] == 0,
    0,
    1 - (discount["revenue_usd"] / discount["gross_revenue_usd"]),
)

discount["item_discount_percent"] = np.where(
    discount["gross_revenue_usd"] == 0,
    0,
    discount["item_discount_usd"] / discount["gross_revenue_usd"],
)

discount["voucher_discount_percent"] = np.where(
    discount["gross_revenue_usd"] == 0,
    0,
    discount["voucher_discount_usd"] / discount["gross_revenue_usd"],
)

In [20]:
discount

,sub_product_line,id_shop_name,henry_category_1,week_id,gross_revenue_usd,revenue_usd,item_discount_usd,voucher_discount_usd,discount_utilization,item_discount_percent,voucher_discount_percent
0,Basic,ID,Bottoms,week1,135.165571,129.316391,0.0,0.000000,0.043274,0.0,0.000000
1,Basic,ID,Bottoms,week2,218.077297,165.349441,0.0,0.000000,0.241785,0.0,0.000000
2,Basic,ID,Bottoms,week3,417.768651,293.162908,0.0,0.000000,0.298265,0.0,0.000000
3,Basic,ID,Bottoms,week4,333.996817,245.689323,0.0,0.000000,0.264396,0.0,0.000000
4,Basic,ID,Bottoms,week5,266.644736,206.069282,0.0,0.000000,0.227177,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
730,Workwear,TH,Tops,week3,1311.181717,987.243534,0.0,4.411523,0.247058,0.0,0.003365
731,Workwear,TH,Tops,week4,884.809040,704.623662,0.0,4.349746,0.203643,0.0,0.004916
732,Workwear,TH,Tops,week5,652.132391,510.149222,0.0,4.167375,0.217721,0.0,0.006390
733,Workwear,TH,Tops,week6,257.958543,225.415301,0.0,0.000000,0.126157,0.0,0.000000


In [21]:
del discount["gross_revenue_usd"]
del discount["revenue_usd"]
del discount["item_discount_usd"]
del discount["voucher_discount_usd"]

In [22]:
discount

,sub_product_line,id_shop_name,henry_category_1,week_id,discount_utilization,item_discount_percent,voucher_discount_percent
0,Basic,ID,Bottoms,week1,0.043274,0.0,0.000000
1,Basic,ID,Bottoms,week2,0.241785,0.0,0.000000
2,Basic,ID,Bottoms,week3,0.298265,0.0,0.000000
3,Basic,ID,Bottoms,week4,0.264396,0.0,0.000000
4,Basic,ID,Bottoms,week5,0.227177,0.0,0.000000
...,...,...,...,...,...,...,...
730,Workwear,TH,Tops,week3,0.247058,0.0,0.003365
731,Workwear,TH,Tops,week4,0.203643,0.0,0.004916
732,Workwear,TH,Tops,week5,0.217721,0.0,0.006390
733,Workwear,TH,Tops,week6,0.126157,0.0,0.000000


In [23]:
discount.groupby(["week_id", "id_shop_name"])[
    "discount_utilization", "item_discount_percent", "voucher_discount_percent"
].mean()

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/ipykernel/__main__.py:3: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  app.launch_new_instance()


discount_utilization  item_discount_percent  \
week_id id_shop_name                                                
week1   ID                        0.179816               0.000000   
        MY                        0.142887               0.000000   
        SG                        0.134639               0.000523   
        TH                        0.173073               0.000000   
week2   ID                        0.174237               0.000000   
        MY                        0.168804               0.000000   
        SG                        0.136885               0.001619   
        TH                        0.156178               0.000000   
week3   ID                        0.176591               0.000000   
        MY                        0.132004               0.000000   
        SG                        0.194422               0.005587   
        TH                        0.174697               0.000000   
week4   ID                        0.150931               0.000000   
        MY                        0.186771               0.000000   
        SG                        0.171490               0.009720   
        TH                        0.165998               0.000000   
week5   ID                        0.184440               0.000000   
        MY                        0.184005               0.000000   
        SG                        0.196238               0.001611   
        TH                        0.213542               0.000000   
week6   ID                        0.226414               0.000000   
        MY                        0.174699               0.000000   
        SG                        0.185393               0.000000   
        TH                        0.180859               0.000000   
week7   ID                        0.218909               0.000000   
        MY                        0.259340               0.000000   
        SG                        0.193627               0.000000   
        TH                        0.231841               0.000000   

                      voucher_discount_percent  
week_id id_shop_name                            
week1   ID                            0.008884  
        MY                            0.000275  
        SG                            0.008545  
        TH                            0.008887  
week2   ID                            0.008562  
        MY                            0.000052  
        SG                            0.004225  
        TH                            0.004234  
week3   ID                            0.013820  
        MY                            0.000063  
        SG                            0.011424  
        TH                            0.010849  
week4   ID                            0.011378  
        MY                            0.000000  
        SG                            0.013205  
        TH                            0.015291  
week5   ID                            0.000836  
        MY                            0.000183  
        SG                            0.030382  
        TH                            0.011402  
week6   ID                            0.006508  
        MY                            0.000000  
        SG                            0.015238  
        TH                            0.000837  
week7   ID                            0.009788  
        MY                            0.000762  
        SG                            0.010734  
        TH                            0.000000

In [24]:
discount.to_csv(
    "s3://hal-bi-bucket/data_science/dfm/offline_clothing_v2/data/deploy_discount_offline_median.csv",
    index=False,
)